[![Homepage](https://img.shields.io/badge/homepage-blueviolet?logo=htmx)](https://www.bendai.org/CUHK-STAT3009/)
[![GitHub](https://img.shields.io/badge/GitHub-black.svg?logo=github)](https://github.com/statmlben/CUHK-STAT3009)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/statmlben/CUHK-STAT3009/blob/main/nb/warmup/NB_Warmup_05_MF_Recap.ipynb)

# NB Warm-up 05 — MF end to end: fit → predict → GridSearchCV

**STAT3009 · Recommender Systems**  
**Suggested time:** 18–20 minutes

> **Workflow warm-up:** use the estimator interface. Fit a provided MF model, predict ratings, then choose $K$ and $\lambda$ with cross-validation.

**Route:** model and API (3 min) → `fit` (3 min) → `predict` (3 min) → `GridSearchCV` (7 min) → refit and exit ticket (3 min). We reuse the 16 ratings from Warm-up 04; no download is needed.

## 1. The MF model in one minute

$$\widehat r_{ui}=\mu+a_u+b_i+p_u^\mathsf{T}q_i.$$

| Question | MF answer |
|---|---|
| **Model** | baseline $\mu+a_u+b_i$ plus the interaction $p_u^\mathsf{T}q_i$ |
| **Hyperparameters** | $K$: number of latent coordinates; $\lambda$: regularization strength |
| **Fitting criterion** | squared error on observed training pairs, plus a penalty on the factors |
| **Learned parameters** | $\mu$, biases $a,b$, and factor matrices $P,Q$ |

We will **not** re-derive ALS in this warm-up. The goal is to practise the same `fit` → `predict` → `GridSearchCV` workflow you will use in the notebook and the midterm.

## 2. The data from Warm-up 04

Each row of `X_ratings` is `[user_id, item_id]`; the matching entry of `y_ratings` is its rating. The full ID vocabulary is 5 users and 4 items, fixed before any fit or CV fold.

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.utils.validation import check_is_fitted

n_users, n_items = 5, 4

X_ratings = np.array([
    [0, 0], [0, 1],
    [1, 0], [1, 1], [1, 2], [1, 3],
    [2, 0], [2, 2], [2, 3],
    [3, 1], [3, 2], [3, 3],
    [4, 0], [4, 1], [4, 2], [4, 3],
])
y_ratings = np.array([
    5., 4.,
    1., 2., 2., 1.,
    4., 4., 5.,
    3., 4., 3.,
    2., 3., 3., 2.,
])

print('X shape:', X_ratings.shape)
print('y shape:', y_ratings.shape)
print('ID vocabulary:', n_users, 'users and', n_items, 'items')
print('observed ratings:', len(y_ratings))

## 3. The provided MF estimator

The class below is **supplied**. You are not asked to implement or analyse ALS here. Read the interface instead:

- the constructor stores $K$, $\lambda$, `n_iters`, and `random_state`;
- `fit(X, y)` learns attributes ending in `_` from the fitting rows;
- `predict(X)` accepts a 2-column array of `[user_id, item_id]` pairs.

The vocabulary sizes `n_users` and `n_items` are fixed before fitting. This matters when a CV fold does not contain every ID.

In [ ]:
class BiasedMF(RegressorMixin, BaseEstimator):
    def __init__(self, n_users, n_items, K=2, lambda_=1.0,
                 n_iters=5, random_state=42):
        self.n_users = n_users
        self.n_items = n_items
        self.K = K
        self.lambda_ = lambda_
        self.n_iters = n_iters
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=int)
        y = np.asarray(y, dtype=float)
        users, items = X[:, 0], X[:, 1]

        self.mu_ = y.mean()
        self.a_ = np.zeros(self.n_users)
        self.b_ = np.zeros(self.n_items)

        rng = np.random.default_rng(self.random_state)
        self.P_ = rng.normal(0, 0.1, size=(self.n_users, self.K))
        self.Q_ = rng.normal(0, 0.1, size=(self.n_items, self.K))

        rows_by_user = [[] for _ in range(self.n_users)]
        rows_by_item = [[] for _ in range(self.n_items)]
        for row, (u, i) in enumerate(X):
            rows_by_user[u].append(row)
            rows_by_item[i].append(row)

        self.seen_users_ = np.array([bool(rows) for rows in rows_by_user])
        self.seen_items_ = np.array([bool(rows) for rows in rows_by_item])
        self.P_[~self.seen_users_] = 0.0
        self.Q_[~self.seen_items_] = 0.0

        for _ in range(self.n_iters):
            # Fix users, update each observed item.
            for i, rows in enumerate(rows_by_item):
                if not rows:
                    continue
                u = users[rows]
                ridge = Ridge(alpha=self.lambda_, fit_intercept=True)
                ridge.fit(self.P_[u], y[rows] - self.mu_ - self.a_[u])
                self.b_[i] = ridge.intercept_
                self.Q_[i] = ridge.coef_

            # Fix items, update each observed user.
            for u, rows in enumerate(rows_by_user):
                if not rows:
                    continue
                i = items[rows]
                ridge = Ridge(alpha=self.lambda_, fit_intercept=True)
                ridge.fit(self.Q_[i], y[rows] - self.mu_ - self.b_[i])
                self.a_[u] = ridge.intercept_
                self.P_[u] = ridge.coef_
        return self

    def predict(self, X):
        check_is_fitted(self, 'mu_')
        X = np.asarray(X, dtype=int)
        users, items = X[:, 0], X[:, 1]
        return (
            self.mu_ + self.a_[users] + self.b_[items]
            + np.sum(self.P_[users] * self.Q_[items], axis=1)
        )

## 4. Step 1 — `fit`

Create

```python
BiasedMF(n_users=n_users, n_items=n_items, K=2, lambda_=1.0,
         n_iters=5, random_state=42)
```

fit it on `X_ratings, y_ratings`, and report the training RMSE. Training RMSE is a **diagnostic**, not the score used to choose $K$ and $\lambda$.

In [ ]:
# Your turn: create the model, fit it, and print the training RMSE.
# mf = ...
# mf.fit(...)
# print(...)

### Solution

In [ ]:
mf = BiasedMF(
    n_users=n_users, n_items=n_items,
    K=2, lambda_=1.0, n_iters=5, random_state=42,
)
mf.fit(X_ratings, y_ratings)

print('settings:', mf.get_params())
print('learned global mean:', round(mf.mu_, 3))
print('user factor shape:', mf.P_.shape)
print('item factor shape:', mf.Q_.shape)
print('training RMSE:', round(root_mean_squared_error(
    y_ratings, mf.predict(X_ratings)
), 3))

## 5. Step 2 — `predict`

Predict the four pairs below. `(0, 0)` and `(3, 1)` are observed, while `(0, 2)` and `(2, 1)` are missing pairs from the same users/items.


In [ ]:
X_new = np.array([
    [0, 0],  # observed pair
    [0, 2],  # missing pair, known user and item
    [2, 1],  # missing pair, known user and item
    [3, 1],  # observed pair
])

# Your turn: predict all four pairs.
# print(...)

### Solution


In [ ]:
predictions = mf.predict(X_new)
print('predictions:', np.round(predictions, 3))


## 6. Step 3 — choose $K$ and $\lambda$ with `GridSearchCV`

Compare `K = 2, 4` and `lambda_ = 0.1, 10.0` with three-fold cross-validation. Use `scoring='neg_root_mean_squared_error'` and `refit=True`.

Before running: why are `P_`, `Q_`, `mu_`, `a_`, and `b_` **not** in `param_grid`? What kind of quantity belongs there instead?

In [ ]:
# Your turn: complete the GridSearchCV workflow.
# search = GridSearchCV(
#     estimator=BiasedMF(n_users=n_users, n_items=n_items,
#                        n_iters=5, random_state=42),
#     param_grid={...},
#     scoring='...',
#     cv=...,
#     refit=True,
# )
# search.fit(X_ratings, y_ratings)

### Solution

In [ ]:
search = GridSearchCV(
    estimator=BiasedMF(n_users=n_users, n_items=n_items,
                       n_iters=5, random_state=42),
    param_grid={'K': [2, 4], 'lambda_': [0.1, 10.0]},
    scoring='neg_root_mean_squared_error',
    cv=3,
    refit=True,
)
search.fit(X_ratings, y_ratings)

for K, lambda_, score in zip(
    search.cv_results_['param_K'],
    search.cv_results_['param_lambda_'],
    search.cv_results_['mean_test_score'],
):
    print(f'K={K}, lambda={lambda_}: CV RMSE={-score:.3f}')

print('selected settings:', search.best_params_)
print(f'best CV RMSE: {-search.best_score_:.3f}')

## 7. Step 4 — read the result and refit

`refit=True` has already fitted `search.best_estimator_` on all 16 observed ratings after CV. Use that model to predict `X_new` again. Compare:

- `search.best_score_`: validation RMSE from CV;
- training RMSE of `best_estimator_`: a fit diagnostic on all 16 rows.

They answer different questions. Neither one is a test-set score.

In [ ]:
best_mf = search.best_estimator_
print('refitted predictions:', np.round(best_mf.predict(X_new), 3))
print('refitted training RMSE:', round(root_mean_squared_error(
    y_ratings, best_mf.predict(X_ratings)
), 3))
print('best CV RMSE:', round(-search.best_score_, 3))

## 8. The workflow map

| Step | Call | Question it answers |
|---|---|---|
| Fit one candidate | `mf.fit(X, y)` | What parameters does this setting learn? |
| Predict | `mf.predict(X_new)` | What rating do we output for these pairs? |
| Compare candidates | `search.fit(X, y)` | Which $K,\lambda$ has the best validation RMSE? |
| Final model | `search.best_estimator_` | Which fitted model do we use after selection? |
| Report | `best_estimator_.predict(X_new)` | What are the final predictions? |

The estimator interface is the same one used by `GridSearchCV`: constructor → `fit` → `predict`. The internals of the model can change without changing this workflow.

## Exit ticket

1. Which MF quantities are chosen before `fit`, and which are learned inside `fit`?
2. What is the difference between `search.best_score_` and the training RMSE of `search.best_estimator_`?
3. Why must each CV fold call `fit` again instead of reusing `mf`'s learned arrays?
4. For a pair with a new user and a known item, which terms of the MF prediction survive?

<details>
<summary><b>After discussion</b></summary>

1. $K$, $\lambda$, `n_iters`, and `random_state` are chosen before fitting. $\mu$, $a$, $b$, $P$, and $Q$ are learned from the fitting rows.
2. `best_score_` is the validation RMSE from CV. The training RMSE is computed on the same 16 rows used by the refitted model; it is usually optimistic and does not measure generalization.
3. Each fold has different fitting rows, so its $\mu$, biases, and factors must be learned without the validation ratings.
4. A new user has $a_u=0$ and $p_u=0$, so only $\mu+b_i$ remains. If both IDs are new, the prediction is $\mu$.

</details>